# 20 — OLS + BioBERT NER Pipeline

**Architecture:**
1. PRIDE API (authoritative structured metadata)
2. BioBERT NER — pre-trained on PubMed, zero training examples needed.
   Extracts Disease, CellLine, OrganismPart entities from abstract+methods.
   Used as Layer 3, only fills columns PRIDE didn't fill.
3. Regex — fills protocol fields (instrument, enzyme, mods, tolerances)
4. OLS normalization at every step — canonical NT=;AC= format
5. Majority fallback with dominance check

**Why BioBERT NER over regex for bio fields:**
- Already knows HUVEC=cell line, serum=biofluid, frontal cortex=brain region
- Understands context: 'HeLa spike-in' ≠ study cell line
- No training needed — pre-trained on millions of PubMed abstracts

## 0. Imports and paths

In [1]:
import os, re, json, time, difflib
from collections import defaultdict, Counter
from pathlib import Path
from functools import lru_cache

import requests
import pandas as pd
from tqdm import tqdm

# BioBERT NER — pre-trained, no fine-tuning needed
try:
    from transformers import pipeline as hf_pipeline, AutoTokenizer, AutoModelForTokenClassification
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print('transformers not available — BioBERT NER will be skipped')

IS_KAGGLE = Path('/kaggle').exists()
if IS_KAGGLE:
    BASE_PATH = Path('/kaggle/input/harmonizing-the-data-of-your-data')
    OUT_PATH  = Path('/kaggle/working/submission_biobert_ols.csv')
    CACHE_PATH = Path('/kaggle/input/sdrf-api-caches')
else:
    BASE_PATH  = Path.cwd().parent / 'data'
    OUT_PATH   = Path.cwd().parent / 'outputs' / 'submission_biobert_ols.csv'
    CACHE_PATH = Path.cwd().parent / 'outputs' / 'kaggle_dataset'

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
TRAIN_SDRF_DIR = BASE_PATH / 'TrainingSDRFs'
SAMPLE_SUB     = BASE_PATH / 'SampleSubmission.csv'
OLS_TIMEOUT    = 10

_pubtext_candidates = [
    BASE_PATH / 'Test_PubText' / 'Test PubText',
    BASE_PATH / 'Test_PubText',
    BASE_PATH / 'TestPubText',
    BASE_PATH / 'Test PubText',
]
TEST_TEXT_DIR = next((p for p in _pubtext_candidates if p.exists()), _pubtext_candidates[0])
print(f'IS_KAGGLE  : {IS_KAGGLE}')
print(f'PubText    : {TEST_TEXT_DIR} exists={TEST_TEXT_DIR.exists()}')
print(f'TrainSDRFs : {TRAIN_SDRF_DIR} exists={TRAIN_SDRF_DIR.exists()}')
print(f'HF_AVAILABLE: {HF_AVAILABLE}')

c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IS_KAGGLE  : False
PubText    : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TestPubText exists=True
TrainSDRFs : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\data\TrainingSDRFs exists=True
HF_AVAILABLE: True


## 1. OLS functions and ontology dicts

In [2]:
_ols_session = requests.Session()
_ols_session.headers.update({'Accept': 'application/json', 'User-Agent': 'SDRF-OLS/1.0'})

@lru_cache(maxsize=2000)
def ols_lookup(term, ontology):
    """Query OLS4 for a term in a specific ontology.
    Returns canonical 'NT=label;AC=accession' string or None.
    ontology: 'uberon', 'ms', 'unimod', 'pride'
    """
    if not term or str(term).strip().lower() in ('not applicable','na','n/a',''):
        return None
    try:
        r = _ols_session.get(
            'https://www.ebi.ac.uk/ols4/api/search',
            params={
                'q': str(term).strip(),
                'ontology': ontology,
                'rows': 1,
                'exact': 'false',
                'fieldList': 'label,obo_id,short_form'
            },
            timeout=OLS_TIMEOUT
        )
        if r.status_code != 200: return None
        docs = r.json().get('response', {}).get('docs', [])
        if not docs: return None
        doc = docs[0]
        label  = doc.get('label', '')
        obo_id = doc.get('obo_id', '') or doc.get('short_form', '')
        if label and obo_id:
            # PSI-MS instruments use AC=;NT= order (convention)
            if ontology == 'ms' and obo_id.startswith('MS:'):
                return f'AC={obo_id};NT={label}'
            return f'NT={label};AC={obo_id}'
        return None
    except Exception:
        return None


def ols_organism(name):
    """Resolve organism to NCBI taxon ID format: '9606 (Homo sapiens)'"""
    # Fast local lookup first
    ORGANISM_ONT = {
        'homo sapiens': '9606 (Homo sapiens)', 'human': '9606 (Homo sapiens)',
        'humans': '9606 (Homo sapiens)',
        'mus musculus': '10090 (Mus musculus)', 'mouse': '10090 (Mus musculus)',
        'mice': '10090 (Mus musculus)', 'murine': '10090 (Mus musculus)',
        'rattus norvegicus': '10116 (Rattus norvegicus)', 'rat': '10116 (Rattus norvegicus)',
        'saccharomyces cerevisiae': '4932 (Saccharomyces cerevisiae)',
        'yeast': '4932 (Saccharomyces cerevisiae)',
        'escherichia coli': '562 (Escherichia coli)', 'e. coli': '562 (Escherichia coli)',
        'e.coli': '562 (Escherichia coli)',
        'drosophila melanogaster': '7227 (Drosophila melanogaster)',
        'danio rerio': '7955 (Danio rerio)', 'zebrafish': '7955 (Danio rerio)',
        'arabidopsis thaliana': '3702 (Arabidopsis thaliana)',
        'sus scrofa': '9823 (Sus scrofa)', 'pig': '9823 (Sus scrofa)',
        'porcine': '9823 (Sus scrofa)',
        'bos taurus': '9913 (Bos taurus)', 'bovine': '9913 (Bos taurus)',
        'gallus gallus': '9031 (Gallus gallus)', 'chicken': '9031 (Gallus gallus)',
        'caenorhabditis elegans': '6239 (Caenorhabditis elegans)',
        'c. elegans': '6239 (Caenorhabditis elegans)',
        'xenopus laevis': '8355 (Xenopus laevis)',
        'macaca mulatta': '9544 (Macaca mulatta)',
        'rabbit': '9986 (Oryctolagus cuniculus)',
        'oryctolagus cuniculus': '9986 (Oryctolagus cuniculus)',
        'dog': '9615 (Canis lupus familiaris)',
    }
    n = str(name).lower().strip()
    for key in sorted(ORGANISM_ONT, key=len, reverse=True):
        if key in n: return ORGANISM_ONT[key]
    return None


def ols_tissue(name):
    """Resolve tissue/organ to UBERON canonical string, with local fallback."""
    # Local fast lookup for common terms
    TISSUE_FAST = {
        'blood plasma': 'NT=blood plasma;AC=UBERON:0001969',
        'plasma': 'NT=blood plasma;AC=UBERON:0001969',
        'blood serum': 'NT=blood serum;AC=UBERON:0001977',
        'serum': 'NT=blood serum;AC=UBERON:0001977',
        'whole blood': 'NT=blood;AC=UBERON:0000178',
        'blood': 'NT=blood;AC=UBERON:0000178',
        'peripheral blood': 'NT=blood;AC=UBERON:0000178',
        'urine': 'NT=urine;AC=UBERON:0001088',
        'cerebrospinal fluid': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
        'csf': 'NT=cerebrospinal fluid;AC=UBERON:0001359',
        'saliva': 'NT=saliva;AC=UBERON:0001836',
        'brain': 'NT=brain;AC=UBERON:0000955',
        'prefrontal cortex': 'NT=prefrontal cortex;AC=UBERON:0000451',
        'frontal cortex': 'NT=frontal cortex;AC=UBERON:0001870',
        'cerebral cortex': 'NT=cerebral cortex;AC=UBERON:0000956',
        'hippocampus': 'NT=hippocampal formation;AC=UBERON:0002421',
        'cerebellum': 'NT=cerebellum;AC=UBERON:0002037',
        'liver': 'NT=liver;AC=UBERON:0002107',
        'lung': 'NT=lung;AC=UBERON:0002048',
        'heart': 'NT=heart;AC=UBERON:0000948',
        'kidney': 'NT=kidney;AC=UBERON:0002113',
        'pancreas': 'NT=pancreas;AC=UBERON:0001264',
        'colon': 'NT=colon;AC=UBERON:0001155',
        'prostate': 'NT=prostate gland;AC=UBERON:0002367',
        'prostate gland': 'NT=prostate gland;AC=UBERON:0002367',
        'breast': 'NT=breast;AC=UBERON:0000310',
        'ovary': 'NT=ovary;AC=UBERON:0000992',
        'spleen': 'NT=spleen;AC=UBERON:0002106',
        'bone marrow': 'NT=bone marrow;AC=UBERON:0002371',
        'adipose tissue': 'NT=adipose tissue;AC=UBERON:0001013',
        'adipose': 'NT=adipose tissue;AC=UBERON:0001013',
        'skeletal muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
        'muscle': 'NT=skeletal muscle;AC=UBERON:0001134',
        'skin': 'NT=skin of body;AC=UBERON:0002097',
        'thymus': 'NT=thymus;AC=UBERON:0002370',
        'lymph node': 'NT=lymph node;AC=UBERON:0000029',
        'testis': 'NT=testis;AC=UBERON:0000473',
        'retina': 'NT=retina;AC=UBERON:0000966',
        'pbmc': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
        'peripheral blood mononuclear': 'NT=peripheral blood mononuclear cell;AC=CL:0000057',
        'platelet': 'NT=platelet;AC=CL:0000233',
        'extracellular vesicle': 'NT=extracellular vesicle;AC=GO:0061695',
        'exosome': 'NT=extracellular vesicle;AC=GO:0061695',
    }
    n = str(name).lower().strip()
    for key in sorted(TISSUE_FAST, key=len, reverse=True):
        if key in n: return TISSUE_FAST[key]
    # Fall back to OLS
    result = ols_lookup(name, 'uberon')
    return result


def ols_instrument(name):
    """Resolve instrument to PSI-MS canonical AC=MS:XXXXXX;NT=Name."""
    INSTRUMENT_FAST = {
        'q exactive hf-x': 'AC=MS:1003027;NT=Q Exactive HF-X',
        'q exactive hf': 'AC=MS:1002523;NT=Q Exactive HF',
        'q exactive plus': 'AC=MS:1002634;NT=Q Exactive Plus',
        'q-exactive plus': 'AC=MS:1002634;NT=Q Exactive Plus',
        'q exactive': 'AC=MS:1001911;NT=Q Exactive',
        'qexactive': 'AC=MS:1001911;NT=Q Exactive',
        'orbitrap astral': 'AC=MS:1003378;NT=Orbitrap Astral',
        'orbitrap fusion lumos': 'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
        'fusion lumos': 'AC=MS:1002732;NT=Orbitrap Fusion Lumos',
        'orbitrap fusion': 'AC=MS:1002416;NT=Orbitrap Fusion',
        'orbitrap eclipse': 'AC=MS:1003029;NT=Orbitrap Eclipse',
        'orbitrap exploris 480': 'AC=MS:1003094;NT=Orbitrap Exploris 480',
        'exploris 480': 'AC=MS:1003094;NT=Orbitrap Exploris 480',
        'ltq orbitrap velos': 'AC=MS:1001742;NT=LTQ Orbitrap Velos',
        'ltq orbitrap elite': 'AC=MS:1001910;NT=LTQ Orbitrap Elite',
        'ltq orbitrap xl': 'AC=MS:1000556;NT=LTQ Orbitrap XL',
        'ltq orbitrap': 'AC=MS:1000449;NT=LTQ Orbitrap',
        'timstof pro 2': 'AC=MS:1003474;NT=timsTOF Pro 2',
        'timstof pro': 'AC=MS:1003231;NT=timsTOF Pro',
        'timstof': 'AC=MS:1002817;NT=timsTOF',
        'triple tof 6600': 'AC=MS:1000931;NT=TripleTOF 6600',
        'triple tof 5600': 'AC=MS:1000931;NT=TripleTOF 5600',
        'triple tof': 'AC=MS:1000931;NT=TripleTOF 6600',
        'impact ii': 'AC=MS:1002817;NT=impact II',
        'synapt g2': 'AC=MS:1002726;NT=Synapt G2-Si',
        'velos pro': 'AC=MS:1001909;NT=LTQ Velos Pro',
    }
    n = str(name).lower().strip()
    # Check for already-normalized AC=;NT= format
    ac = re.search(r'AC=(MS:\d+)', name)
    nt = re.search(r'NT=([^;]+)', name)
    if ac and nt:
        return f'AC={ac.group(1).strip()};NT={nt.group(1).strip()}'
    for key in sorted(INSTRUMENT_FAST, key=len, reverse=True):
        if key in n: return INSTRUMENT_FAST[key]
    # Fall back to OLS MS ontology
    result = ols_lookup(name, 'ms')
    return result


def fmt_label(n):
    n = str(n).lower().strip()
    if any(x in n for x in ['label free','label-free','lfq','label_free','unlab']):
        return 'AC=MS:1002038;NT=label free sample'
    if 'tmt' in n:
        m = re.search(r'tmt[\s\-]?(\d+)', n)
        p = m.group(1) if m else '6'
        acc = {'2':'MS:1002456','6':'MS:1002453','10':'MS:1002454',
               '11':'MS:1002454','16':'MS:1003998','18':'MS:1003999'}
        return f'AC={acc.get(p,"MS:1002453")};NT=TMT{p}plex'
    if 'itraq' in n:
        m = re.search(r'itraq[\s\-]?(\d+)', n)
        p = m.group(1) if m else '4'
        return f"AC={'MS:1001985' if p=='4' else 'MS:1002519'};NT=iTRAQ{p}plex"
    if 'silac' in n: return 'AC=MS:1002791;NT=SILAC'
    if 'dimethyl' in n: return 'AC=MS:1002457;NT=Dimethyl'
    return str(n)


# Quick OLS test
print('Testing OLS4...')
test = ols_lookup('blood serum', 'uberon')
print(f'  blood serum → {test}')
test2 = ols_lookup('Q Exactive HF', 'ms')
print(f'  Q Exactive HF → {test2}')
print('OLS ready.')

Testing OLS4...
  blood serum → NT=blood serum;AC=UBERON:0001977
  Q Exactive HF → AC=MS:1002523;NT=Q Exactive HF
OLS ready.


## 2. Training data

In [3]:
sample_sub  = pd.read_csv(SAMPLE_SUB)
id_cols     = ['ID','PXD','Raw Data File','Usage']
target_cols = [c for c in sample_sub.columns
               if c not in id_cols and 'Unnamed' not in c]
all_base    = set(re.sub(r'\.\d+$','',c) for c in target_cols)

def _strip_wrapper(col):
    m = re.match(r'(?:characteristics|comment|factor\s*value)\[(.+?)\]', col, re.I)
    return m.group(1) if m else col

def _find_col(col, df_cols):
    if col in df_cols: return col
    base = re.sub(r'\.\d+$','',col)
    if base in df_cols: return base
    stripped = _strip_wrapper(base)
    if stripped in df_cols: return stripped
    return None

col_counters = {col: Counter() for col in target_cols}
col_vocab    = defaultdict(set)
train_files  = []
if TRAIN_SDRF_DIR.exists():
    train_files = list(TRAIN_SDRF_DIR.glob('*.tsv')) + list(TRAIN_SDRF_DIR.glob('*.csv'))

train_pxd_sdrf = {}
for fp in train_files:
    sep = '\t' if fp.suffix == '.tsv' else ','
    try: df = pd.read_csv(fp, low_memory=False, sep=sep)
    except: continue
    pxd = fp.stem.replace('Harmonized_','').replace('_cleaned.sdrf','').split('.')[0]
    pxd_vals = {}
    for col in target_cols:
        mc = _find_col(col, set(df.columns))
        if mc:
            vals = df[mc].dropna().astype(str)
            vals = vals[~vals.str.lower().isin(['not applicable','n/a','na',''])]
            col_counters[col].update(vals.tolist())
            col_vocab[re.sub(r'\.\d+$','',col)].update(vals.tolist())
            uniq = list(vals.unique())
            if uniq: pxd_vals[col] = uniq
    train_pxd_sdrf[pxd] = pxd_vals

global_modes = {}
non_na_ratio = {}
n_train = max(len(train_files), 1)
for col in target_cols:
    total = sum(col_counters[col].values())
    if total > 0:
        global_modes[col] = col_counters[col].most_common(1)[0][0]
        non_na_ratio[col] = total / n_train
    else:
        global_modes[col] = 'Not Applicable'
        non_na_ratio[col] = 0.0

# Columns that are too experiment-specific for majority fallback
NO_FALLBACK = {
    'Characteristics[SyntheticPeptide]','Characteristics[PooledSample]',
    'Characteristics[Bait]','Characteristics[TumorSize]',
    'Characteristics[GrowthRate]','Characteristics[SamplingTime]',
    'Characteristics[Time]','Characteristics[Compound]',
    'Characteristics[ConcentrationOfCompound]','Characteristics[Treatment]',
    'Characteristics[DiseaseTreatment]','Characteristics[Depletion]',
    'Characteristics[CellPart]','Characteristics[Age]','Characteristics[BMI]',
    'Characteristics[AncestryCategory]',
    # Study-biology columns: never broadcast a training-majority value across all PXDs
    'Characteristics[DevelopmentalStage]',  # 'fetal' dominates training; wrong for most test studies
    'Characteristics[CellLine]',            # 'HEK293T' dominates training; wrong for tissue studies
    'FactorValue[Bait]','FactorValue[CellPart]','FactorValue[Treatment]',
    'FactorValue[Disease]','FactorValue[Compound]',
    'FactorValue[ConcentrationOfCompound].1','FactorValue[GeneticModification]',
    'FactorValue[Temperature]','FactorValue[FractionIdentifier]',
}

print(f'Train SDRFs : {len(train_files)}')
print(f'Target cols : {len(target_cols)}')
print(f'NO_FALLBACK : {len(NO_FALLBACK)} columns excluded from fallback')


Train SDRFs : 103
Target cols : 77
NO_FALLBACK : 27 columns excluded from fallback


## 3. PRIDE cache

In [4]:
# Load pre-fetched PRIDE cache — no internet needed on Kaggle
PRIDE_CACHE_PATH = CACHE_PATH / 'pride_cache.json'
OLS_CACHE_PATH   = CACHE_PATH / 'ols_cache.json'

with open(PRIDE_CACHE_PATH) as f:
    PRIDE_CACHE = json.load(f)
print(f'PRIDE cache: {len(PRIDE_CACHE)} PXDs')

with open(OLS_CACHE_PATH) as f:
    OLS_CACHE_DICT = json.load(f)
print(f'OLS cache  : {len(OLS_CACHE_DICT)} terms')


def ols_lookup_cached(term, ontology):
    key = f'{ontology}::{str(term).lower().strip()}'
    return OLS_CACHE_DICT.get(key)


def _ols_with_cache(term, ontology, fast_fn):
    norm = fast_fn(term)
    if norm: return norm
    return ols_lookup_cached(term, ontology)


def fetch_pride_cached(pxd):
    data = PRIDE_CACHE.get(pxd, {}).get('pride_api', {})
    if not data: return {}
    out = defaultdict(list)
    for o in data.get('organisms', []):
        v = ols_organism(o.get('name',''))
        if v: out['Characteristics[Organism]'].append(v)
    for op in (data.get('organisms_part') or data.get('tissues') or []):
        name = op.get('name',''); acc = op.get('accession','')
        if name and name.lower() not in ('not available','n/a',''):
            v = ols_tissue(name) or ols_lookup_cached(name,'uberon')
            if v: out['Characteristics[OrganismPart]'].append(v)
            elif acc: out['Characteristics[OrganismPart]'].append(f'NT={name};AC={acc}')
    for dis in data.get('diseases', []):
        name = dis.get('name','')
        if name and name.lower() not in ('not available','n/a','none','normal',''):
            out['Characteristics[Disease]'].append(name)
    for inst in data.get('instruments', []):
        name = inst.get('name',''); acc = inst.get('accession','')
        v = ols_instrument(name) or ols_lookup_cached(name,'ms')
        if v: out['Comment[Instrument]'].append(v)
        elif acc: out['Comment[Instrument]'].append(f'AC={acc};NT={name}')
    for qm in data.get('quantification_methods', []):
        v = fmt_label(qm.get('name',''))
        if v: out['Characteristics[Label]'].append(v)
    return {k: list(dict.fromkeys(v)) for k,v in out.items() if v}


print('PRIDE cache loaded.')

PRIDE cache: 15 PXDs
OLS cache  : 89 terms
PRIDE cache loaded.


## 4. BioBERT NER

Uses `d4data/biomedical-ner-all` — pre-trained on PubMed, recognises:
`DISEASE`, `CHEMICAL`, `CELL_LINE`, `CELL_TYPE`, `DNA`, `RNA`, `PROTEIN`

**Key design decisions:**
- Only fills columns PRIDE didn't fill (PRIDE is authoritative)
- Requires entity to appear in abstract (high confidence) not just methods
- Maps NER labels to SDRF columns via controlled vocabulary
- OLS normalization applied to every NER hit

In [6]:
# Lazy NER pipeline -- d4data/biomedical-ner-all
# aggregation_strategy='simple' automatically merges ## subword fragments
# (e.g. cytomegalovirus is NOT split into cyt/##ome/##gal/##ovirus).
# We keep the ## guard as a belt-and-suspenders safety net.
_NER_PIPE = None
NER_MODEL  = 'd4data/biomedical-ner-all'

def _get_ner_pipe():
    global _NER_PIPE
    if _NER_PIPE is None and HF_AVAILABLE:
        _NER_PIPE = hf_pipeline(
            'ner',
            model=NER_MODEL,
            tokenizer=NER_MODEL,
            aggregation_strategy='simple',   # merges ## subword fragments automatically
            device=-1,                       # CPU
        )
    return _NER_PIPE

_SKIP_DISEASE = {
    'disease','cancer','tumor','tumour','normal','healthy',
    'viral','infection','infections','bacteria','bacterial',
    'fungal','syndrome','disorder','inflammation',
}

def ner_extract(abstract: str, methods: str, pride_filled: set) -> dict:
    # Run BioBERT NER; return {column: [values]}.
    # Only fills columns not already covered by pride_filled.
    # aggregation_strategy='simple' handles ## subword merging at pipeline level.
    pipe = _get_ner_pipe()
    if pipe is None or not abstract:
        return {}
    out = defaultdict(list)

    def add(col, val):
        if val and val not in out[col]: out[col].append(val)

    for text, is_abstract in [(abstract, True), (methods, False)]:
        if not text:
            continue
        # Truncate to stay inside BERT 512-token limit (~2000 chars is safe)
        text_chunk = text[:2000]
        try:
            entities = pipe(text_chunk)
        except Exception:
            continue

        for ent in entities:
            label    = ent.get('entity_group', '').upper()
            word     = ent.get('word', '').strip()
            score    = ent.get('score', 0)
            if score < 0.80:
                continue          # low-confidence entity
            word_low = word.lower()

            # Belt-and-suspenders: drop any residual ## fragments
            if word.startswith('##'):
                continue
            # Drop 1-2 char tokens (abbreviations / noise)
            if len(word) <= 2:
                continue

            # Disease
            if label in ('DISEASE', 'DISEASE_DISORDER') and is_abstract:
                if 'Characteristics[Disease]' not in pride_filled:
                    if word_low not in _SKIP_DISEASE:
                        norm = DISEASE_NORM.get(word_low) or DISEASE_NORM.get(word) or word
                        add('Characteristics[Disease]', norm)

            # Cell Line
            elif label == 'CELL_LINE' and is_abstract:
                if 'Characteristics[CellLine]' not in pride_filled:
                    add('Characteristics[CellLine]', word)

            # Cell Type
            elif label == 'CELL_TYPE' and is_abstract:
                if 'Characteristics[CellType]' not in pride_filled:
                    add('Characteristics[CellType]', word)

    return dict(out)

print('BioBERT NER functions defined.')
print('NER model will be loaded on first call (lazy loading).')


BioBERT NER functions defined.
NER model will be loaded on first call (lazy loading).


## 5. Text utilities and filename parsers

In [24]:
        (re.compile(r'\b(fetal|fetus|foetal)(?!\s+(?:bovine|calf))\b',re.I),'fetal'),

((re.compile(r'\b(fetal|fetus|foetal)(?!\s+(?:bovine|calf))\b',
             re.IGNORECASE|re.UNICODE),
  'fetal'),)

In [19]:
def parse_fraction(rf):
    for p in [
        r'[_\-\.](fx?|fr|frac(?:tion)?)[_\-\.\s]?(\d{1,3})(?=[_\-\.]|$)',
        r'[_\-](\d{1,3})of\d+[_\-\.]',
        r'fraction(\d{1,3})',
    ]:
        m = re.search(p, str(rf), re.I)
        if m:
            n = m.group(2) if m.lastindex and m.lastindex >= 2 else m.group(1)
            if n and n.isdigit() and 1 <= int(n) <= 200:
                return str(int(n))
    return None

def parse_biol_rep(rf):
    for p in [
        r'[_\-]biolrep[_\-]?(\d+)',
        r'[_\-]br(\d+)[_\-\.]',
        r'[_\-]rep(\d+)[_\-\.]',
        r'[_\-]r(\d{1,2})[_\-\.]',
    ]:
        m = re.search(p, str(rf), re.I)
        if m and m.group(1).isdigit() and 1 <= int(m.group(1)) <= 50:
            return str(int(m.group(1)))
    return None

def parse_label_from_filename(rf):
    rf_up = str(rf).upper()
    m = re.search(r'TMT(PRO|18|16|11|10|6|2)', rf_up)
    if m:
        pmap = {'PRO':'16','18':'18','16':'16','11':'11','10':'10','6':'6','2':'2'}
        amap = {'18':'MS:1003999','16':'MS:1003998','11':'MS:1002454',
                '10':'MS:1002454','6':'MS:1002453','2':'MS:1002456'}
        p = pmap.get(m.group(1),'6')
        return f'AC={amap[p]};NT=TMT{p}plex'
    if 'TMT' in rf_up: return 'AC=MS:1002453;NT=TMT6plex'
    if re.search(r'SILAC|_H_|_HVY|_L_|_LGT', rf_up): return 'AC=MS:1002791;NT=SILAC'
    if re.search(r'LFQ|LABELFREE|_LF_', rf_up): return 'AC=MS:1002038;NT=label free sample'
    return None

def _filename_stem(fn: str) -> str:
    """
    Strip extension and trailing run/scan/fraction index so that replicate files
    collapse to the same stem key while different sample groups stay apart.

    'COVID_Serum_rep1_001.raw'  -> 'covid_serum_rep1'
    'Ctrl_rep1_001.raw'         -> 'ctrl_rep1'
    'PXD_f003.mzML'             -> 'pxd'
    """
    s = re.sub(r'\.\w{1,6}$', '', fn)                     # drop extension
    s = re.sub(r'[_.\-]?\d{2,6}$', '', s)                 # trailing run index
    s = re.sub(r'[_.\-]f(?:r|rac(?:tion)?)?\d+$', '', s,  # _fr001 / _frac03
               flags=re.I)
    return s.lower()

print('Filename parsers ready.')


Filename parsers ready.


## 6. Load test papers

In [20]:
test_docs = {}
pxd_to_raws = {}
for _, row in sample_sub.iterrows():
    pxd_to_raws.setdefault(row['PXD'],[]).append(row['Raw Data File'])

if TEST_TEXT_DIR.exists():
    for fp in sorted(TEST_TEXT_DIR.glob('*.json')):
        pxd = fp.stem.split('_')[0]
        try:
            d = json.loads(fp.read_text(encoding='utf-8', errors='replace'))
            if d: test_docs[pxd] = d
        except: pass

print(f'Test papers : {len(test_docs)}')
print(f'Test PXDs   : {len(pxd_to_raws)}')
for pxd, d in test_docs.items():
    print(f'  {pxd}: {len(get_text(d)):,} chars')

Test papers : 16
Test PXDs   : 15
  PubText: 0 chars
  PXD004010: 9,632 chars
  PXD016436: 6,855 chars
  PXD019519: 43,569 chars
  PXD025663: 18,591 chars
  PXD040582: 18,378 chars
  PXD050621: 12,514 chars
  PXD061009: 32,999 chars
  PXD061090: 18,055 chars
  PXD061136: 14,323 chars
  PXD061195: 8,117 chars
  PXD061285: 32,587 chars
  PXD062014: 28,053 chars
  PXD062469: 15,042 chars
  PXD062877: 33,750 chars
  PXD064564: 17,873 chars


## 7. Main pipeline

In [21]:
final_sub = pd.read_csv(SAMPLE_SUB, dtype=str).copy()
for col in target_cols:
    final_sub[col] = 'Not Applicable'

def fuzzy_snap(value, base_col, cutoff=0.82):
    if not value or base_col not in col_vocab: return value
    matches = difflib.get_close_matches(value, list(col_vocab[base_col]), n=1, cutoff=cutoff)
    return matches[0] if matches else value

# Bio columns where PRIDE is authoritative over everything else
BIO_COLS = {
    'Characteristics[Organism]', 'Characteristics[OrganismPart]',
    'Characteristics[Disease]',  'Characteristics[MaterialType]',
    'Characteristics[CellLine]',
}

for pxd, pxd_df in tqdm(final_sub.groupby('PXD'), desc='PXDs'):
    idx       = pxd_df.index
    raw_files = pxd_to_raws[pxd]
    pub_dict  = test_docs.get(pxd, {})
    abstract  = get_abstract(pub_dict) if pub_dict else ''
    methods   = get_methods(pub_dict)  if pub_dict else ''

    pxd_vals = defaultdict(list)

    def pxd_add(col, val):
        if not val: return
        v = str(val).strip()
        if v.lower() in ('not applicable','na','n/a','','null','none'): return
        base = re.sub(r'\.\d+$', '', col)
        if base == 'Characteristics[Organism]':
            v = ols_organism(v) or v
        elif base == 'Characteristics[OrganismPart]':
            if not v.startswith('NT='): v = ols_tissue(v) or v
        elif base == 'Comment[Instrument]':
            if not v.startswith('AC=MS:'): v = ols_instrument(v) or v
        snapped = fuzzy_snap(v, base)
        if snapped not in pxd_vals[col]: pxd_vals[col].append(snapped)

    # ── Layer 0: Training overlap ──────────────────────────────────────────
    if pxd in train_pxd_sdrf:
        for col, vals in train_pxd_sdrf[pxd].items():
            for v in (vals or []): pxd_add(col, v)

    # ── Layer 1: PRIDE cache ───────────────────────────────────────────────
    pride_data = fetch_pride_cached(pxd)
    for col, vals in pride_data.items():
        for v in (vals or []): pxd_add(col, v)

    # Track what PRIDE filled — these columns are locked for bio fields
    pride_filled = {re.sub(r'\.\d+$','',c) for c,v in pxd_vals.items() if v}

    # ── Layer 2: BioBERT NER ───────────────────────────────────────────────
    # Only runs if there is paper text; only fills what PRIDE didn't
    if pub_dict and HF_AVAILABLE:
        ner_vals = ner_extract(abstract, methods, pride_filled)
        for col, vals in ner_vals.items():
            for v in (vals or []): pxd_add(col, v)
        # Update pride_filled to include NER hits for protocol layer
        ner_filled = {re.sub(r'\.\d+$','',c) for c,v in ner_vals.items() if v}
    else:
        ner_filled = set()

    # ── Layer 3: Protocol regex ────────────────────────────────────────────
    # Bio columns locked by PRIDE or NER; protocol columns always from regex
    bio_filled = pride_filled | ner_filled
    if pub_dict:
        for col, vals in regex_extract(pub_dict).items():
            base = re.sub(r'\.\d+$','',col)
            if base in BIO_COLS and base in bio_filled:
                continue  # already filled by authoritative source
            if isinstance(vals, list):
                for v in vals: pxd_add(col, v)
            else:
                pxd_add(col, vals)

    # ── Layer 4: Majority fallback (dominance check) ───────────────────────
    filled_bases = {re.sub(r'\.\d+$','',c) for c in pxd_vals}
    for col in target_cols:
        base = re.sub(r'\.\d+$','',col)
        if base in filled_bases: continue
        if col in NO_FALLBACK: continue
        total = sum(col_counters[col].values())
        if total > 0:
            top_val, top_count = col_counters[col].most_common(1)[0]
            top_ratio = top_count / total
            if non_na_ratio.get(col,0.0) > 0.80 and top_ratio > 0.80:
                pxd_add(col, top_val)

    # ── Modification slots ─────────────────────────────────────────────────
    mods = list(dict.fromkeys(pxd_vals.pop('Characteristics[Modification]',[])))
    for i, mod in enumerate(mods):
        slot = 'Characteristics[Modification]' if i==0 else f'Characteristics[Modification].{i}'
        pxd_vals[slot] = [mod]

    # ── Spatial kernel: build stem-group rank for every file in this PXD ──
    # Files that share a stem (after stripping extension + run index) form one
    # condition group. When a bio column has multiple candidate values extracted
    # at PXD level, we route by stem rank instead of by file index, so all
    # replicates of the same condition receive the same metadata value.
    _stem_order: dict = {}
    for _rf in raw_files:
        _s = _filename_stem(str(_rf))
        if _s not in _stem_order:
            _stem_order[_s] = len(_stem_order)

    # ── Per-file assignment ────────────────────────────────────────────────
    for i, (row_idx, raw_file) in enumerate(zip(idx, raw_files)):
        fraction = parse_fraction(raw_file)
        biol_rep = parse_biol_rep(raw_file)
        fn_label = parse_label_from_filename(raw_file)
        if fraction: final_sub.at[row_idx,'Comment[FractionIdentifier]'] = fraction
        if biol_rep: final_sub.at[row_idx,'Characteristics[BiologicalReplicate]'] = biol_rep
        if fn_label and final_sub.at[row_idx,'Characteristics[Label]']=='Not Applicable':
            final_sub.at[row_idx,'Characteristics[Label]'] = fn_label
        _stem_rank = _stem_order[_filename_stem(str(raw_file))]
        for col in target_cols:
            if final_sub.at[row_idx,col] != 'Not Applicable': continue
            base = re.sub(r'\.\d+$','',col)
            vals = pxd_vals.get(col) or pxd_vals.get(base) or []
            vals = [v for v in vals if str(v).strip().lower() not in ('not applicable','')]
            if not vals: continue
            # Bio cols with multiple candidates: route by stem-group rank so
            # that all files in the same condition share the same value instead
            # of alternating blindly via round-robin.
            if len(vals) > 1 and base in BIO_COLS:
                final_sub.at[row_idx,col] = vals[_stem_rank % len(vals)]
            else:
                final_sub.at[row_idx,col] = vals[i % len(vals)]

# ── Cleanup ────────────────────────────────────────────────────────────────
final_sub = final_sub.fillna('Not Applicable')
for col in target_cols:
    mask = final_sub[col].astype(str).str.strip().isin(
        ['nan','None','[]','','null','not available','TextSpan','Text Span','not applicable'])
    final_sub.loc[mask,col] = 'Not Applicable'

# Usage column: the sample submission has 'Text Span' as a placeholder; clear it.
# Training SDRFs have no Usage column, so we cannot infer the correct value.
# Not Applicable is safer than Text Span (scorer skips all-NA columns).
if 'Usage' in final_sub.columns:
    final_sub['Usage'] = 'Not Applicable'

# FractionIdentifier artifact cleanup
for pxd, grp in final_sub.groupby('PXD'):
    fracs = grp['Comment[FractionIdentifier]'].unique()
    if len(fracs)==1 and str(fracs[0]).strip() in ('1','Not Applicable'):
        final_sub.loc[grp.index,'Comment[FractionIdentifier]'] = 'Not Applicable'

# Spacing fix
for col in target_cols:
    final_sub[col] = final_sub[col].astype(str).str.replace(r';\s+',';',regex=True)

# Disease normalization
DISEASE_FIX = {
    "Alzheimer's disease":'Alzheimer disease',
    "Parkinson's disease":'Parkinson disease',
    'Lung cancer':'lung carcinoma','lung cancer':'lung carcinoma',
    'Prostate cancer':'prostate carcinoma',
    'Osteoarthritis':'osteoarthritis',
    'Glioblastoma':'glioblastoma',
}
final_sub['Characteristics[Disease]'] = final_sub['Characteristics[Disease]'].replace(DISEASE_FIX)

# Organism format
ORG_FMT = {
    'Homo sapiens (human)':'9606 (Homo sapiens)',
    'Mus musculus (mouse)':'10090 (Mus musculus)',
    'Rattus norvegicus (rat)':'10116 (Rattus norvegicus)',
    'Bos taurus (bovine)':'9913 (Bos taurus)',
    'Escherichia coli':'562 (Escherichia coli)',
}
final_sub['Characteristics[Organism]'] = final_sub['Characteristics[Organism]'].apply(
    lambda x: ORG_FMT.get(str(x).strip(), x))

final_sub.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'Shape : {final_sub.shape}')


PXDs: 100%|██████████| 15/15 [00:19<00:00,  1.32s/it]


Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_biobert_ols.csv
Shape : (1659, 81)


## 7b. Spatial Filename-Stem Healer

**Why this works with near-zero training data (non-parametric approach):**

The extractor layers (PRIDE, BioBERT, Regex) may only tag a fraction of rows per study — e.g. BioBERT might catch
`COVID-19` on 68 of 1,376 rows in PXD061195. Those 68 rows are the "bright pixels" in a feature map
where all 1,376 rows share nearly identical filenames. This layer "bleeds" the confirmed signal
into the dark pixels using only the test set's own internal structure. No weights to train, no risk of
overfitting to the 103 training examples.

**Three feature map channels:**
1. **Filename-stem clusters** (the spatial kernel): files sharing a stem key (after stripping extension + run index) form a group.  
2. **Confidence anchor** (`!= 'Not Applicable'`): only anchored values propagate — hallucinates can't spread if they are outvoted.  
3. **Negative constraints**: `MaterialType = biofluid` cannot bleed into a cell-line-named stem, and vice versa.


In [22]:
import re
from collections import Counter, defaultdict

# ──────────────────────────────────────────────────────────────────────────────
# Which columns to heal.
# Bio columns are the high-value targets; protocol cols (Instrument, Label, etc.)
# are already PXD-uniform from the extraction layers so skipping them avoids
# accidentally overwriting correct stem-routed values.
# ──────────────────────────────────────────────────────────────────────────────
_HEAL_BASES = {
    'Characteristics[Disease]',
    'Characteristics[OrganismPart]',
    'Characteristics[MaterialType]',
    'Characteristics[CellLine]',
    'Characteristics[CellType]',
    'Characteristics[Organism]',
    'Characteristics[Sex]',
    'Comment[Instrument]',
    'Characteristics[Label]',
}
_SPATIAL_COLS = [c for c in final_sub.columns
                 if re.sub(r'\.\d+$', '', c) in _HEAL_BASES]

_NA_SET = {'not applicable', 'nan', '', 'none', 'null'}

# _filename_stem is defined in cell 14 (filename parsers) — no duplicate needed.

# ── Feature Map Channel 2: Confidence Anchor (any non-NA value) ──────────────
# Unlike Gemini's 'AC=' check, every confirmed value — disease name, tissue,
# sex — acts as an anchor. This makes Disease/CellLine propagate too.

def _majority(vals: list) -> str:
    """Return the most common value; break ties alphabetically for stability."""
    c = Counter(vals)
    max_count = c.most_common(1)[0][1]
    candidates = sorted(v for v, n in c.items() if n == max_count)
    return candidates[0]


# ── Feature Map Channel 3: Negative Constraints ──────────────────────────────
_BIOFLUID_PAT = re.compile(
    r'\b(serum|plasma|urine|csf|saliva|balf|bronchoalveolar|sweat)\b', re.I)
_CELLLINE_PAT = re.compile(
    r'\b(hela|hek|mef|u2os|mcf7|a549|jurkat|k562|huvec|cho|pc3|thp1)\b', re.I)


def _neg_constraint_ok(stem: str, col: str, val: str) -> bool:
    """Return False if spreading val into stem would violate a hard constraint."""
    if col != 'Characteristics[MaterialType]':
        return True
    v = val.lower()
    if v == 'biofluid' and _CELLLINE_PAT.search(stem):
        return False
    if v == 'cell line' and _BIOFLUID_PAT.search(stem):
        return False
    return True


# ── Main Spatial Healer (safety-net pass) ────────────────────────────────────
# The primary spatial routing now happens inside the per-file assignment loop
# in cell 18 (stem-group routing). This pass catches any residual gaps that
# remain after that loop — e.g. single-file PXDs or columns the kernel missed.
def spatial_heal(df: pd.DataFrame, heal_cols: list) -> tuple:
    """
    For each PXD:
      1. Group files by filename stem (spatial kernel).
      2. Collect the majority non-NA anchor value per stem group per column.
      3. Fill every 'Not Applicable' cell whose stem has an anchor.
      4. Apply negative constraints before writing.

    Returns (healed_df, stats_dict).
    """
    healed = df.copy()
    stats = defaultdict(int)   # col → n_cells_filled

    for pxd, grp in df.groupby('PXD'):
        idx   = grp.index.tolist()
        stems = [_filename_stem(f) for f in grp['Raw Data File'].astype(str)]

        for col in heal_cols:
            vals = grp[col].astype(str).tolist()

            # Build stem → [confirmed values] pool
            stem_pool: dict = defaultdict(list)
            for stem, v in zip(stems, vals):
                if v.strip().lower() not in _NA_SET:
                    stem_pool[stem].append(v.strip())

            if not stem_pool:
                continue

            # Propagate: fill NA cells from their stem's majority vote
            for row_idx, stem, v in zip(idx, stems, vals):
                if v.strip().lower() not in _NA_SET:
                    continue           # already has a value — never overwrite
                if stem not in stem_pool:
                    continue           # no anchor in this stem group
                winner = _majority(stem_pool[stem])
                if _neg_constraint_ok(stem, col, winner):
                    healed.at[row_idx, col] = winner
                    stats[col] += 1

    return healed, dict(stats)


# ── Run ───────────────────────────────────────────────────────────────────────
final_sub, heal_stats = spatial_heal(final_sub, _SPATIAL_COLS)

# Re-save (the main pipeline already saved once; overwrite with healed version)
final_sub.to_csv(OUT_PATH, index=False)

total_healed = sum(heal_stats.values())
print(f'Spatial healer (safety-net): {total_healed:,} additional cells filled')
print(f'Saved → {OUT_PATH}')
if heal_stats:
    print('\nPer-column fill counts:')
    for col, n in sorted(heal_stats.items(), key=lambda x: -x[1]):
        print(f'  {col:<55} +{n:,}')


Spatial healer (safety-net): 0 additional cells filled
Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_biobert_ols.csv


## 8. Validation

In [23]:
label_cols = [c for c in final_sub.columns
              if c not in ('ID','PXD','Raw Data File','Usage')]
rows = [(c,(final_sub[c]!='Not Applicable').sum()) for c in label_cols]
rows.sort(key=lambda x: -x[1])

print(f'{"Column":<55} {"Filled":>7} {"Pct":>6}')
print('-'*72)
for col, n in rows:
    if n > 0:
        print(f'{col:<55} {n:>7} {n/len(final_sub)*100:>5.1f}%')

filled = sum(1 for _,n in rows if n>0)
print(f'\nTotal filled: {filled} / {len(rows)}')

# Spot check suspicious columns
print('\n=== Spot check ===')
for col in ['Characteristics[SyntheticPeptide]','Characteristics[Bait]',
            'Characteristics[PooledSample]','Characteristics[Organism]',
            'Characteristics[OrganismPart]','Comment[Instrument]',
            'Characteristics[Disease]']:
    if col in final_sub.columns:
        vc = final_sub[col].value_counts().head(2)
        na = (final_sub[col]=='Not Applicable').sum()
        print(f'\n{col} (NA={na}):')
        for v,n in vc.items():
            print(f'  {str(v)[:70]}: {n}')

Column                                                   Filled    Pct
------------------------------------------------------------------------
Characteristics[BiologicalReplicate]                       1659 100.0%
Characteristics[Organism]                                  1659 100.0%
Comment[Instrument]                                        1659 100.0%
Comment[MS2MassAnalyzer]                                   1659 100.0%
Characteristics[Modification]                              1635  98.6%
Characteristics[Modification].1                            1635  98.6%
Characteristics[Modification].2                            1635  98.6%
Characteristics[Modification].3                            1635  98.6%
Characteristics[Modification].4                            1635  98.6%
Characteristics[Modification].5                            1635  98.6%
Characteristics[Modification].6                            1635  98.6%
Characteristics[CleavageAgent]                             1634  98.5%
Char